<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>


# Testing - Integration

## Integration Testing vs Unit Testing

Up to this point, every test we have written has been a **unit test**, a test that isolates a single function and verifies it behaves correctly on its own. We passed in controlled fixture data, we mocked external dependencies, and we checked one piece of logic at a time. Unit tests answer the question: *does this function do what it's supposed to?*

But in a real data pipeline, functions don't run in isolation. `extract_from_csv` feeds into `handle_types`, which feeds into `impute_values`, which feeds into `add_features`, and eventually the result is written back to disk by `load_to_csv`. Each of those functions might pass its unit tests perfectly yet the pipeline as a whole could still fail. A type conversion might produce a dtype that the next function doesn't expect. A column might get renamed in one step and break a downstream reference. The output path might not get created. These are **integration failures**: bugs that only surface when components interact.

**Integration tests** exist to catch exactly these problems. Rather than testing a single function with fake data, an integration test wires the real components together and runs them as a connected chain, ideally against realistic data and real infrastructure (file systems, cloud storage, databases). The tradeoffs are straightforward:

| | Unit Tests | Integration Tests |
|---|---|---|
| **Scope** | One function in isolation | Multiple components working together |
| **Data** | Small fixtures, mocks, controlled inputs | Production-like or real data sources |
| **Speed** | Fast (milliseconds) | Slower (seconds to minutes, may hit network/disk) |
| **What they catch** | Logic errors within a single function | Wiring errors, contract mismatches, IO failures |
| **When they fail** | Something is wrong with *this* function | Something is wrong with how components *connect* |
| **How many you need** | Many, cover every edge case per function | Fewer, cover the critical paths through the system |

A practical rule of thumb for data engineers: **unit tests validate your transformations, integration tests validate your pipeline**. You want a large base of fast unit tests that catch logic bugs early, and a smaller set of integration tests that confirm the whole flow, extract, transform, load, actually produces the right output file with the right schema.

The test below is a true integration test. It uses the production config, hits the real data source, runs every preprocessing step, and writes to a temporary output path. If this test passes, you have high confidence the pipeline will work when deployed.

---

## `test_pipeline_integration.py`

This script provides a true end-to-end integration test for the entire data engineering pipeline, using production configuration values from `config.py`. Unlike the other pipeline tests which use small fixture data from `conftest.py`, this test runs the full `run_de_pipeline` function against the real data source (e.g. an S3 URL), making it a realistic validation that the pipeline works in conditions close to production. It still uses pytest's `tmp_path` fixture for the output destination to avoid polluting the project directory with test artefacts.

- **Individual Test Cases:**
    - **`test_pipeline_e2e_flow`:**
        - **Objective:** Verifies that the full pipeline, from extracting raw data via the production config path, through all preprocessing steps, to writing the final output executes successfully and produces a correctly structured output file.
        - **Assertions:**
            - `pytest.fail(f"Pipeline failed to execute end-to-end: {e}")`: Wrapped in a `try-except` block, this catches any unexpected exception during pipeline execution and converts it into an explicit test failure with a descriptive message, rather than letting the test crash with a raw traceback.
            - `assert os.path.exists(test_output_path)`: A DevOps-level IO check confirming that the pipeline actually wrote a physical file to disk, catching cases where the pipeline might complete silently without producing output.
            - `assert col in processed_df.columns` (looped over `config.COLUMNS_TO_KEEP`): A schema integrity check that reads the output file back and verifies every expected column from the production config is present in the final DataFrame, ensuring no columns were dropped or renamed during processing.
            - `assert pd.api.types.is_numeric_dtype(processed_df['churn_binary'])`: Confirms that the target encoding step produced a numeric `churn_binary` column, which is essential for any downstream modelling or analysis that consumes this pipeline's output.

In [ ]:
##test_pipeline_integration.py
import os
import pytest
import pandas as pd
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))

from src import config
from src.pipeline import run_de_pipeline

def test_pipeline_e2e_flow(tmp_path, raw_churn_df, pipeline_params):
     
     """
    Recommended Scenario: Tests the full flow from extraction to local save.
    Uses tmp_path (pytest fixture) to avoid polluting your project with test files.
    """
     
    # --- 1. ARRANGE ---
    # Create a temporary directory for our "mock" environment
     input_dir = tmp_path / "input"
     output_dir = tmp_path / "output"
     input_dir.mkdir()
     output_dir.mkdir()

    # Save our fixture to a temporary CSV (the "raw" data)
     input_file_path = input_dir / "raw_data.csv"
     raw_churn_df.to_csv(input_file_path, index=False)
     output_file_path = output_dir / "processed_data.csv"
    
        
    # 2. Run the actual pipeline using production config values
     try:
        # --- 2. ACT ---
        run_de_pipeline(
            input_path=str(input_file_path),
            output_path=str(output_file_path),
            target=pipeline_params['target'],
            num_cols=pipeline_params['numeric_cols'],
            cat_cols=pipeline_params['categorical_cols'],
            final_cols=pipeline_params['final_columns']
        )
     except Exception as e:
        pytest.fail(f"Pipeline failed to execute end-to-end: {e}")

    # 3. Verify physical file creation (DevOps/IO Check)
     assert os.path.exists(output_file_path), "Pipeline finished but no output file was written."

    # 4. Schema Integrity Check (Data Engineering Best Practice)
     processed_df = pd.read_csv(output_file_path)
    
    # Check that all requested columns exist
     for col in pipeline_params['final_columns']:
        assert col in processed_df.columns, f"Missing expected column: {col}"
    
    # Check that churn_binary was created and is numeric
     assert pd.api.types.is_numeric_dtype(processed_df['churn_binary']), "churn_binary should be numeric."